# OGC application-package generator
Turn an existing repository into a CWL workflow whose steps follow the data produced and consumed by its commands. We build a source graph, group related functions, interpret their scientific roles, then generate and validate CWL and an example job.

Use `from generate import generate` and `cwl, job = generate(repo_url, container_image)` for the complete pipeline. The cells below explain each stage and let you inspect its graph. See [generate.py](generate.py) for the callable function.

### Setup
Load the graph, semantic-analysis and validation helpers, plus local `.env` configuration. Static analysis gathers the evidence; later Gemini calls interpret it and generate the documents.


In [1]:
import json
import logging
import os
from functools import partial
from pathlib import Path

from IPython.display import display

from analysis import analyze_repo, cluster_graph, cluster_view, container_metadata
from semantic import gemini, label_clusters, infer_io_connections
from generate import load_env, generate_package

load_env()

log = logging.getLogger("generator")


### Configuration
The repository supplies the code evidence; the container supplies the runtime for its existing commands. Optional documentation adds scientific context, example input paths and small-run settings that the code alone may not explain.


In [2]:
REPO_URL = "https://github.com/sunnydean/lightning2earthcare/tree/main"
ENTRYPOINTS = None  # Discover main guards / console scripts, or supply ["module.function"].
CONTAINER_IMAGE = "ghcr.io/sunnydean/lightning2earthcare:latest"
DOC_FILES = []  # Optional local text files describing the workflow or small-run data.
# Replace these small-run settings when changing repositories.
EXTRA_DOCS = "Use 2025-07-31, MTG-I1 only, frame A, and a 10-minute half-window.\n"
EXTRA_DOCS += "\n\n".join(Path(path).read_text() for path in DOC_FILES)
MODEL = os.environ.get("GEMINI_MODEL", "gemini-flash-latest")  # Tracks the latest Flash release; override if needed.
MAX_ATTEMPTS = 5
REQUIRE_GRAPH = True  # False: graph mismatches become recommendations.

ask = partial(gemini, model=MODEL)


## 1. Build the code graph
Start at runnable entry points and follow statically resolvable calls to build a directed graph of functions. Nodes retain source and resource-access observations for later interpretation; unresolved calls remain observations rather than guessed connections.

Command ownership records which executable uses each function, helping distinguish workflow steps from internal operations. Container metadata adds how those commands start and which working directory they expect. This stage reads source and metadata without executing the application.


In [3]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
graph = analyze_repo(REPO_URL, ENTRYPOINTS)
graph.graph.update(extra_docs=EXTRA_DOCS, container_image=CONTAINER_IMAGE)
graph.graph["container"] = container_metadata(CONTAINER_IMAGE)
print("Container:", graph.graph["container"])


2026-10-12 12:37:49,025 INFO GET https://api.github.com/repos/sunnydean/lightning2earthcare/commits/main
2026-10-12 12:37:49,383 INFO GET https://api.github.com/repos/sunnydean/lightning2earthcare/git/trees/6e180fc385ab5b6e3ba11c8f76e68d2c308bbe6e?recursive=1
2026-10-12 12:37:49,675 INFO GET https://raw.githubusercontent.com/sunnydean/lightning2earthcare/a36cbda325920ee78575390df12ea110a8053c54/Dockerfile
2026-10-12 12:37:49,959 INFO GET https://raw.githubusercontent.com/sunnydean/lightning2earthcare/a36cbda325920ee78575390df12ea110a8053c54/README.md
2026-10-12 12:37:50,199 INFO GET https://raw.githubusercontent.com/sunnydean/lightning2earthcare/a36cbda325920ee78575390df12ea110a8053c54/cwl/helper_create_earthcare_catalog.py
2026-10-12 12:37:50,466 INFO GET https://raw.githubusercontent.com/sunnydean/lightning2earthcare/a36cbda325920ee78575390df12ea110a8053c54/lightning2ec/README.md
2026-10-12 12:37:50,688 INFO GET https://raw.githubusercontent.com/sunnydean/lightning2earthcare/a36cbda3

Container: {'image': 'ghcr.io/sunnydean/lightning2earthcare:latest', 'digest': 'sha256:a1fbb206411c0b3c3ac5d074cf2ff48d60480a1b1b5741a0c1016c5cc683b0ad', 'platform': 'linux/arm64', 'entrypoint': ['/app/entrypoint.sh', 'python', '-m', 'lightning2ec.cli'], 'cmd': ['--help'], 'working_dir': '/app'}


### Before clustering
Inspect the structure we found: each node is a function, and arrows show calls or command registration. Heavy outlines identify entry points and exposed commands; expand the details to see the functions behind them.


In [4]:
display(cluster_view(graph))


### Group related functions
Group functions that are close along call paths into candidate scientific substeps. Agglomerative clustering merges nearby groups up to a distance threshold, so the number of clusters emerges from the graph.

Command ownership and disconnected components keep unrelated operations separate. Raise `distance_threshold` for broader groups or lower it for more detail; `separate_entrypoints` keeps executable boundaries visible.


In [5]:
CLUSTER_OPTIONS = {
    "distance_threshold": 4.0,   # Higher: fewer/larger groups; lower: finer groups.
    "separate_entrypoints": True,  # Keep entry points and exposed commands separate.
}

clusters = cluster_graph(graph, **CLUSTER_OPTIONS)
log.info("%d functions, %d directed connections, %d clusters, %d call observations",
         len(graph), graph.number_of_edges(), len(clusters),
         sum(len(node["calls"]) for _, node in graph.nodes(data=True)))
print("Entry points:", graph.graph["entrypoints"])
for cluster in clusters:
    print(cluster["id"], "commands:", cluster["commands"], "used by:", cluster["used_by"], cluster["members"])


2026-10-12 12:37:56,122 INFO 56 functions, 58 directed connections, 10 clusters, 964 call observations


Entry points: ['lightning2ec.cli.main']
cluster_1 commands: [] used by: [] ['lightning2ec.cli.main']
cluster_2 commands: ['lightning2ec'] used by: ['lightning2ec'] ['lightning2ec.cli.run_pipeline']
cluster_3 commands: [] used by: ['lightning2ec'] ['lightning2ec.clustering._filter_by_quality', 'lightning2ec.clustering.parent_clustering', 'lightning2ec.clustering.subclustering', 'lightning2ec.collocation.collocate_glm_to_ec', 'lightning2ec.collocation.collocate_li_to_ec', 'lightning2ec.collocation.summarize_vs_cpr', 'lightning2ec.earthcare_io.fetch_earthcare_data', 'lightning2ec.lightning_io._get_datastore', 'lightning2ec.lightning_io._iter_hours', 'lightning2ec.lightning_io._parse_glm_timefield', 'lightning2ec.lightning_io._to_datetime', 'lightning2ec.lightning_io.download_li', 'lightning2ec.lightning_io.load_merge_glm', 'lightning2ec.lightning_pipeline.process_one_source', 'lightning2ec.parallax._get_parallax_shift_xyz', 'lightning2ec.parallax._get_satellite_elevation', 'lightning2ec.p

### After clustering
The same view now collapses each cluster into one node, making the larger structure easier to inspect. Arrows connect groups whose functions are connected, and the details retain every member function. These groups are starting points for scientific interpretation.


In [6]:
display(cluster_view(graph))


## 2. Interpret operations and data flow
Gemini reads the source evidence to name each cluster, describe its purpose and split groups that mix distinct operations. It identifies resource reads and writes, then uses them to connect command outputs to the inputs that consume them.

Existing commands become executable CWL steps; internal clusters become documented substeps. External resources remain input requirements during repairs. Command rendering and source review check the proposed interfaces before fixing them as the basis for generation.


In [7]:
clusters = label_clusters(graph, ask)
connections = infer_io_connections(graph, ask)
print(json.dumps(connections, indent=2))
print("I/O uncertainties:", graph.graph["io_uncertainties"])


2026-10-12 12:37:56,141 INFO Gemini request: gemini-flash-latest (157710 prompt characters)
2026-10-12 12:38:21,660 INFO Gemini usage: {'promptTokenCount': 48352, 'candidatesTokenCount': 5829, 'totalTokenCount': 54181, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 48352}], 'serviceTier': 'standard'}
2026-10-12 12:38:21,662 INFO cluster_1: CLI root entrypoint (1 functions)
2026-10-12 12:38:21,662 INFO cluster_2: Pipeline CLI command (1 functions)
2026-10-12 12:38:21,663 INFO cluster_3: Lightning data collocation processing (29 functions)
2026-10-12 12:38:21,663 INFO cluster_4: Storm catalogue file processor (3 functions)
2026-10-12 12:38:21,664 INFO cluster_5: Create storm catalogue CLI (1 functions)
2026-10-12 12:38:21,664 INFO cluster_6: Collocation runtime orchestration (13 functions)
2026-10-12 12:38:21,664 INFO cluster_7: NetCDF and JSON to Parquet conversion (4 functions)
2026-10-12 12:38:21,664 INFO cluster_8: Parquet postprocessing CLI (1 functions)
2026-10-12 12:38

[
  {
    "source": "lightning2ec",
    "source_output": "lightning_groups_dir",
    "target": "create_storm_catalogue",
    "target_input": "lightning_groups_dir",
    "artifact": "lightning_groups"
  },
  {
    "source": "lightning2ec",
    "source_output": "track_counts_dir",
    "target": "create_storm_catalogue",
    "target_input": "track_counts_dir",
    "artifact": "track_counts"
  },
  {
    "source": "create_storm_catalogue",
    "source_output": "catalogue",
    "target": "post_process_nc_to_parquet",
    "target_input": "catalogue",
    "artifact": "catalogue.json"
  },
  {
    "source": "lightning2ec",
    "source_output": "collocation_dir",
    "target": "post_process_nc_to_parquet",
    "target_input": "source_dir",
    "artifact": "collocation_data"
  },
  {
    "source": "post_process_nc_to_parquet",
    "source_output": "parquet_dir",
    "target": "create_stac_catalogue",
    "target_input": "parquet_dir",
    "artifact": "parquet_data"
  },
  {
    "source": "lightn

### After interpretation
The view picks up the scientific labels, descriptions and inferred artifact links now stored in the graph. Grey arrows show code connections; dashed orange arrows show data passed between commands. Inspect these links and the cluster details to understand the proposed workflow.


In [8]:
display(cluster_view(graph))


## 3. Generate and validate the package
Code builds the CWL tools and connections directly from the reviewed graph. Gemini supplies short descriptions and example-job values. Cluster descriptions are attached as `s:hasPart` metadata so the scientific substeps remain visible within their owning commands.

Validation checks CWL/OGC structure, job types and agreement with the graph's commands, inputs, outputs, connections and substeps. Findings feed back into a bounded repair loop; passing documents become the `cwl` and `job` strings. These are static checks: runtime behavior and small-run compute remain unverified.

Set `REQUIRE_GRAPH = False` to report graph mismatches as recommendations without blocking acceptance. CWL/OGC and job validation remain required; recommendations are logged.


In [9]:
cwl, job = generate_package(graph, ask, max_attempts=MAX_ATTEMPTS, require_graph=REQUIRE_GRAPH)
cwl, job


2026-10-12 12:41:25,783 INFO Candidate 1/5
2026-10-12 12:41:25,784 INFO Gemini request: gemini-flash-latest (13588 prompt characters)
2026-10-12 12:42:05,254 INFO Gemini usage: {'promptTokenCount': 3750, 'candidatesTokenCount': 4061, 'totalTokenCount': 13159, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3750}], 'thoughtsTokenCount': 5348, 'serviceTier': 'standard'}
2026-10-12 12:42:05,503 INFO Candidate 1: PASS


("cwlVersion: v1.2\n$namespaces:\n  s: https://schema.org/\ns:softwareVersion: 1.0.0\n$graph:\n- class: Workflow\n  id: main\n  label: EarthCARE and Lightning Collocation Pipeline\n  doc: Collocates EarthCARE and satellite lightning data, generating storm catalogues,\n    Parquet datasets, and STAC catalogues.\n  inputs:\n    credentials_file:\n      type: File\n      label: Credentials File\n      doc: MAAP and EUMETSAT credentials file; placeholder /REPLACE_ME/ must be replaced.\n    start_date:\n      type: string\n      label: Start Date\n      doc: Processing start date in YYYY-MM-DD format.\n    end_date:\n      type: string\n      label: End Date\n      doc: Processing end date in YYYY-MM-DD format.\n    earthcare_catalog:\n      type: Directory?\n      label: EarthCARE Catalog\n      doc: Optional directory containing EarthCARE catalog.json metadata.\n    products:\n      type:\n      - 'null'\n      - type: array\n        items: string\n      default:\n      - MSI_COP_2A\n    

### Save the results
Write the validated strings as CWL and job YAML for inspection and use with a workflow runner. Any placeholder input paths still need to be supplied before execution.


In [10]:
Path("package.cwl").write_text(cwl)
Path("job.yml").write_text(job)


183